In [4]:
import pandas as pd
import numpy as np
import warnings
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

class CADDiagnosticPipeline:
    """
    Multi-Paradigm Machine Learning Pipeline for Coronary Artery Disease (CAD).
    Incorporates Rule-Based, Unsupervised, Discriminative, Generative, and Sequential modeling.
    """

    def __init__(self):
        print("--- Initializing MediPredict AI Diagnostic Pipeline ---")
        self._initialize_models()
        self._generate_and_train()
        print("--- System Training Complete & Ready ---")

    def _initialize_models(self):

        self.pca = PCA(n_components=2, random_state=42)
        self.kmeans = KMeans(n_clusters=2, random_state=42)
        self.gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42)

        self.dt_clf = DecisionTreeClassifier(max_depth=4, criterion='entropy', random_state=42)
        self.mlp_clf = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=2500, alpha=1.5, random_state=42)

        self.vectorizer = CountVectorizer()
        self.lda = LatentDirichletAllocation(n_components=2, random_state=42)
        self.nb_clf = MultinomialNB()

        self.memm_clf = LogisticRegression(random_state=42, solver='lbfgs')
        self.hmm_transition = None
        self.hmm_emission = None

    def _train_find_s(self, df):
        positive_cases = df[df['Target'] == 1].drop('Target', axis=1).values
        if len(positive_cases) == 0: return ['?'] * (len(df.columns) - 1)
        hypothesis = positive_cases[0].copy()
        for case in positive_cases[1:]:
            for i in range(len(hypothesis)):
                if hypothesis[i] != case[i]: hypothesis[i] = '?'
        return hypothesis

    def _generate_and_train(self):

        data_A = [
            ['High', 'Yes', 'High', 1],
            ['Normal', 'No', 'Normal', 0],
            ['High', 'No', 'High', 1],
            ['Normal', 'Yes', 'High', 1]
        ]
        self.df_A = pd.DataFrame(data_A, columns=['BP', 'Smoker', 'Cholesterol', 'Target'])
        self.learned_hypothesis = self._train_find_s(self.df_A)

        np.random.seed(42)
        bmi = np.random.normal(26, 4, 1000)
        age = np.random.normal(50, 15, 1000)
        sugar = np.random.normal(110, 25, 1000)
        prior = np.random.randint(0, 2, 1000)
        vitals_raw = np.column_stack((bmi, age, sugar))

        vitals_pca = self.pca.fit_transform(vitals_raw)
        self.kmeans.fit(vitals_pca)
        gmm_clusters = self.gmm.fit_predict(vitals_pca)

        risk = (bmi * 1.5) + (age * 1.2) + (sugar * 0.8) + (prior * 20)
        y_B = np.where(risk > np.median(risk),
                       np.random.choice([0, 1], p=[0.10, 0.90], size=1000),
                       np.random.choice([0, 1], p=[0.90, 0.10], size=1000))

        X_B = np.column_stack((vitals_pca, prior, gmm_clusters))
        X_train, X_test, y_train, y_test = train_test_split(X_B, y_B, test_size=0.2, random_state=42)

        self.dt_clf.fit(X_train, y_train)
        self.mlp_clf.fit(X_train, y_train)

        print("\n[System Evaluation: Model Diagnostics]")
        nn_preds = self.mlp_clf.predict(X_test)
        print(f"Neural Network Base Accuracy: {accuracy_score(y_test, nn_preds)*100:.1f}%")
        print("Classification Report (MLP):\n", classification_report(y_test, nn_preds, target_names=["Low Risk", "High Risk"]))

        texts = [
            "patient complains of severe chest pain shortness of breath prior_1 vitals_1",
            "routine checkup all vitals normal healthy prior_0 vitals_0",
            "urgent review needed high blood pressure abnormal prior_1 vitals_1",
            "patient is doing well no issues reported clear prior_0 vitals_0"
        ]
        y_C = [1, 0, 1, 0]
        X_C = self.vectorizer.fit_transform(texts)
        self.lda.fit(X_C)
        self.nb_clf.fit(X_C, y_C)

        seq_prev_state = np.random.choice([0, 1], size=1000, p=[0.7, 0.3])
        seq_obs = np.where(seq_prev_state == 1,
                           np.random.choice([0, 1], size=1000, p=[0.2, 0.8]),
                           np.random.choice([0, 1], size=1000, p=[0.85, 0.15]))
        seq_curr_state = np.where((seq_prev_state + seq_obs) > 0,
                                  np.random.choice([0, 1], size=1000, p=[0.3, 0.7]),
                                  np.random.choice([0, 1], size=1000, p=[0.95, 0.05]))

        X_memm = np.column_stack((seq_prev_state, seq_obs))
        self.memm_clf.fit(X_memm, seq_curr_state)

        self.hmm_transition = np.array([[0.85, 0.15], [0.10, 0.90]])
        self.hmm_emission = np.array([[0.80, 0.20], [0.30, 0.70]])

    def predict_find_s(self, instance):
        for h, val in zip(self.learned_hypothesis, instance):
            if h != '?' and h != val: return "Does NOT match CAD profile"
        return "MATCHES High-Risk CAD Profile"

    def predict_hmm(self, observation, prev_state=0):
        prior_state = np.array([1.0, 0.0]) if prev_state == 0 else np.array([0.0, 1.0])
        state_probs = prior_state.dot(self.hmm_transition) * self.hmm_emission[:, observation]
        state_probs = state_probs / np.sum(state_probs)
        return np.argmax(state_probs), state_probs[1]

    def predict_memm(self, observation, prev_state=0):

        probs = self.memm_clf.predict_proba([[prev_state, observation]])[0]
        return np.argmax(probs), probs[1]

    def predict_crf(self, observation, prev_state=0):
        node_pot = np.array([[0.8, 0.2], [0.3, 0.7]])
        edge_pot = np.array([[0.9, 0.1], [0.2, 0.8]])
        score_0 = node_pot[observation][0] * edge_pot[prev_state][0]
        score_1 = node_pot[observation][1] * edge_pot[prev_state][1]
        Z = score_0 + score_1
        probs = np.array([score_0 / Z, score_1 / Z])
        return np.argmax(probs), probs[1]


if __name__ == "__main__":
    pipeline = CADDiagnosticPipeline()

    while True:
        print("\n" + "="*75)
        print("NEW PATIENT ENTRY: ADVANCED CAD SCREENING")
        print("="*75)

        print("\n[Module 1: Baseline CAD Concept Learning]")
        bp_input = input("Enter Blood Pressure (High/Normal) or 'quit' to exit: ")
        if bp_input.strip().lower() == 'quit':
            print("\n" + "*"*60)
            print("SYSTEM SHUTDOWN INITIATED.")
            print("*"*60 + "\n")
            break

        bp = bp_input.capitalize()
        smoker = input("Smoker? (Yes/No): ").capitalize()
        chol = input("Enter Cholesterol (High/Normal): ").capitalize()

        user_symptoms = [bp, smoker, chol]
        fs_result = pipeline.predict_find_s(user_symptoms)
        find_s_binary = 1 if "MATCHES" in fs_result else 0
        print(f" -> Current Find-S Diagnosis: {fs_result}")

        print("\n[Modules 2 & 3: Metabolic Vitals (PCA, Clustering, NN, DT)]")
        try:
            bmi_input = float(input("Enter BMI (e.g., 28.5): "))
            age_input = float(input("Enter Age: "))
            sugar_input = float(input("Enter Blood Sugar: "))

            raw_vitals = np.array([[bmi_input, age_input, sugar_input]])
            user_pca = pipeline.pca.transform(raw_vitals)

            user_kmeans = pipeline.kmeans.predict(user_pca)[0]
            user_gmm = pipeline.gmm.predict(user_pca)[0]

            print(f" -> PCA Components: {user_pca[0][0]:.2f}, {user_pca[0][1]:.2f}")
            print(f" -> Unsupervised Grouping (GMM Soft Cluster): Phenotype {user_gmm}")

            user_features = np.column_stack((user_pca, [[find_s_binary]], [[user_gmm]]))

            dt_probs = pipeline.dt_clf.predict_proba(user_features)[0]
            nn_probs = pipeline.mlp_clf.predict_proba(user_features)[0]

            dt_pred, nn_pred = np.argmax(dt_probs), np.argmax(nn_probs)
            vitals_binary = nn_pred

            print(f" -> Decision Tree Risk: {'High Risk' if dt_pred == 1 else 'Low Risk'} ({dt_probs[dt_pred]*100:.1f}%)")
            print(f" -> Neural Network Risk: {'High Risk' if nn_pred == 1 else 'Low Risk'} ({nn_probs[nn_pred]*100:.1f}%)")

        except ValueError:
            vitals_binary = 0
            print(" -> Invalid numerical input. Defaulting to baseline vitals risk.")

        print("\n[Module 4: Clinical Notes Triage (Generative NLP)]")
        user_note = input("Enter Doctor's Note: ")
        enriched_note = f"{user_note} prior_{find_s_binary} vitals_{vitals_binary}"

        try:
            note_vec = pipeline.vectorizer.transform([enriched_note])
            topic_dist = pipeline.lda.transform(note_vec)[0]
            print(f" -> LDA Topic Modeling: Extracted Clinical Topic {np.argmax(topic_dist)}")

            nb_probs = pipeline.nb_clf.predict_proba(note_vec)[0]
            nb_pred = np.argmax(nb_probs)
            print(f" -> Naive Bayes Text Flag: {'URGENT' if nb_pred == 1 else 'ROUTINE'} ({nb_probs[nb_pred] * 100:.1f}%)")
        except Exception:
            print(" -> System encountered unknown vocabulary.")

        print("\n[Module 5: Latent Disease Progression (Sequence Models)]")

        hmm_s, hmm_p = pipeline.predict_hmm(vitals_binary, prev_state=0)
        memm_s, memm_p = pipeline.predict_memm(vitals_binary, prev_state=0)
        crf_s, crf_p = pipeline.predict_crf(vitals_binary, prev_state=0)

        print(f" -> HMM (Generative): Likelihood of CAD Progression: {hmm_p*100:.1f}%")
        print(f" -> MEMM (Discriminative Local): Likelihood of CAD Progression: {memm_p*100:.1f}%")
        print(f" -> CRF (Discriminative Global): Likelihood of CAD Progression: {crf_p*100:.1f}%")

        print("\n--- Pipeline Analysis Complete. Ready for next patient. ---")

--- Initializing MediPredict AI Diagnostic Pipeline ---

[System Evaluation: Model Diagnostics]
Neural Network Base Accuracy: 80.5%
Classification Report (MLP):
               precision    recall  f1-score   support

    Low Risk       0.87      0.77      0.81       111
   High Risk       0.75      0.85      0.80        89

    accuracy                           0.81       200
   macro avg       0.81      0.81      0.80       200
weighted avg       0.81      0.81      0.81       200

--- System Training Complete & Ready ---

NEW PATIENT ENTRY: ADVANCED CAD SCREENING

[Module 1: Baseline CAD Concept Learning]
Enter Blood Pressure (High/Normal) or 'quit' to exit: normal
Smoker? (Yes/No): no
Enter Cholesterol (High/Normal): normal
 -> Current Find-S Diagnosis: Does NOT match CAD profile

[Modules 2 & 3: Metabolic Vitals (PCA, Clustering, NN, DT)]
Enter BMI (e.g., 28.5): 24.9
Enter Age: 21
Enter Blood Sugar: 100
 -> PCA Components: -9.82, -30.16
 -> Unsupervised Grouping (GMM Soft Cluster)